# Featuresmith Tutorial: 02 — Complete Dataset Review Walkthrough

Deep dive into Featuresmith's Review Engine — exploring the 8 automated reviewers, category filtering, finding severities, and remediation guidance.

---


## 1. Why Automated Dataset Code Reviews Matter
Just as software engineers perform code reviews before merging pull requests, ML engineers must perform dataset code reviews before training models. Featuresmith's Review Engine runs 8 specialized reviewers to evaluate dataset health deterministically.

### The 8 Automated Reviewers
1. **Schema Health Reviewer**: Evaluates structural consistency and column naming.
2. **Data Types Reviewer**: Detects text types, numeric types, and type mismatches.
3. **Missing Values Reviewer**: Identifies column missingness ratios and null spikes.
4. **Duplicate Records Reviewer**: Checks for duplicate row entries.
5. **Constant Columns Reviewer**: Finds zero-variance and empty columns.
6. **High Cardinality Reviewer**: Flags categorical columns with excessive unique values.
7. **Basic Statistics Reviewer**: Analyzes distribution skewness and kurtosis anomalies.
8. **Leakage Risk Reviewer**: Evaluates target correlations, identifier shapes, and timestamp anomalies.

### Step 1: Load Dataset & Run Complete Review

In [1]:
import os

import featuresmith as fs

data_path = os.path.join("..", "data", "processed", "sales.csv")
dataset = fs.load(data_path)
review_res = fs.review(dataset)

print(f"Total Sections Evaluated : {len(review_res.sections)}")
print(f"Overall Summary          : {review_res.overall_summary}")

Total Sections Evaluated : 8
Overall Summary          : 4 of 8 sections passed with 5 finding(s) identified across the review.


### Step 2: Inspect Review Sections & Findings

In [2]:
for section in review_res.sections:
    sev_str = (
        section.severity.value
        if hasattr(section.severity, "value")
        else str(section.severity)
    )
    print(f"[{sev_str.upper():<8}] {section.title} ({len(section.findings)} findings)")
    for finding in section.findings:
        print(
            f"     - Column: {finding.column_name or 'dataset':<15} | {finding.title}"
        )

[CRITICAL] Schema Health (1 findings)
     - Column: return_reason   | Fully empty column 'return_reason'
[WARNING ] Constant Columns (1 findings)
     - Column: store_version   | Constant column 'store_version'
[WARNING ] Basic Statistics (2 findings)
     - Column: sales_amount    | High skewness in column 'sales_amount'
     - Column: sales_amount    | High kurtosis in column 'sales_amount'
[INFO    ] Data Types (1 findings)
     - Column: order_id        | Text column 'order_id'
[PASSED  ] Missing Values (0 findings)
[PASSED  ] Duplicate Rows (0 findings)
[PASSED  ] High Cardinality (0 findings)
[PASSED  ] Leakage Detection (0 findings)


### Best Practices & Common Mistakes
- **Best Practice**: Always run `fs.review()` before training baseline models to catch silent structural issues.
- **Common Mistake**: Ignoring `INFO` severity findings like text columns (`name`, `ticket`) that require specialized NLP tokenization or embedding preprocessing.